# Experiment: Erythroid Metabolism Endpoint Addendum

Objective:
- Decide when ATP/glycolysis or related metabolism readouts are worth adding.
- Keep core HbF, maturation, viability, alpha/autophagy, and hemolysis endpoints first.
- Reject metabolism add-ons that create treatment or procurement scope.

In [ ]:
from __future__ import annotations

OPTIONAL = "metabolism_addon_optional"
HOLD = "metabolism_addon_hold"
REJECT = "metabolism_addon_reject"
CORE_FIRST = "core_gate_first"

CORE_FIELDS = [
    "hbf_or_hbg",
    "maturation",
    "viability",
    "alpha_or_autophagy",
    "hemolysis_or_membrane_safety",
]

ADDON_FIELDS = [
    "metabolism_endpoint_named",
    "raw_data_label",
    "low_marginal_cost",
    "timeline_clear",
    "does_not_replace_core",
]

BLOCKED_SCOPE = [
    "treatment_claim",
    "dose_or_patient_action",
    "trial_screening",
    "purchase",
    "import",
    "procurement",
]

In [ ]:
def classify_metabolism_addon(offer: dict[str, bool]) -> dict[str, object]:
    """Classify metabolism endpoint add-ons for a preclinical quote."""
    blocked = [field for field in BLOCKED_SCOPE if offer.get(field)]
    if blocked:
        return {"decision": REJECT, "reason": "blocked_scope", "fields": blocked}

    missing_core = [field for field in CORE_FIELDS if not offer.get(field)]
    if missing_core:
        return {
            "decision": CORE_FIRST,
            "reason": "core_gate_incomplete",
            "fields": missing_core,
        }

    if offer.get("does_not_replace_core") is False:
        return {
            "decision": REJECT,
            "reason": "replaces_core_endpoint",
            "fields": ["does_not_replace_core"],
        }

    missing_addon = [field for field in ADDON_FIELDS if not offer.get(field)]
    if missing_addon:
        return {
            "decision": HOLD,
            "reason": "addon_details_incomplete",
            "fields": missing_addon,
        }

    return {"decision": OPTIONAL, "reason": "cheap_additive_context", "fields": []}

## Synthetic Add-On Cases

These examples describe endpoint labels, not real lab offers or treatment advice.

In [ ]:
complete = {field: True for field in CORE_FIELDS + ADDON_FIELDS}
complete.update({field: False for field in BLOCKED_SCOPE})

cases = {
    "core_missing": complete | {"hemolysis_or_membrane_safety": False},
    "cheap_additive": complete,
    "cost_unclear": complete | {"low_marginal_cost": False},
    "replaces_core": complete | {"does_not_replace_core": False},
    "treatment_scope": complete | {"treatment_claim": True},
}

results = {name: classify_metabolism_addon(case) for name, case in cases.items()}
results

In [ ]:
expected = {
    "core_missing": CORE_FIRST,
    "cheap_additive": OPTIONAL,
    "cost_unclear": HOLD,
    "replaces_core": REJECT,
    "treatment_scope": REJECT,
}

decision_summary = {name: result["decision"] for name, result in results.items()}
assert decision_summary == expected, decision_summary
decision_summary

## Decision

- Keep metabolism endpoints optional.
- Do not discuss them until the core quote gate is intact.
- Reject add-ons that create treatment, trial, purchase, import, or procurement scope.